# UrbanStyle.ltd — RFM Kliendisegmenteerimine

**Kuupäev:** 2025-02-28  
**Autor:** Vladimir G  
**Eesmärk:** Segmenteerida UrbanStyle'i kliendid RFM analüüsi abil, et anda Markole tegevussoovitused turunduskampaaniate jaoks.

**Pipeline struktuur:**
- Roll A: Data Loading — andmete laadimine ja liitmine
- Roll B: Data Cleaning — puhastamine ja valideerimine  
- Roll C: RFM Analysis — Recency, Frequency, Monetary + segmendid
- Roll D: Visualization — 3 diagrammi + äritõlgendus

---
## Roll A: Data Loading — Andmete laadimine ja liitmine

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

print('Teegid laetud.')

In [ ]:
# Laadi CSV failid
df_sales = pd.read_csv('sales.csv')
df_customers = pd.read_csv('customers.csv')

print(f'df_sales shape: {df_sales.shape}')
print(f'df_customers shape: {df_customers.shape}')
print('\ndf_sales veerud:', list(df_sales.columns))
print('df_customers veerud:', list(df_customers.columns))

In [ ]:
# Kontrolli müügiandmete üldpilti
print('df_sales dtypes:')
print(df_sales.dtypes)
df_sales.head()

In [ ]:
# Liida tabelid customer_id põhjal (LEFT JOIN)
df = pd.merge(
    df_sales,
    df_customers[['customer_id', 'email', 'first_name', 'last_name', 'city']],
    on='customer_id',
    how='left'
)

print(f'Liidatud DataFrame shape: {df.shape}')
print(f'Veerud: {list(df.columns)}')
df.head()

---
## Roll B: Data Cleaning — Puhastamine ja valideerimine

In [ ]:
print(f'Esialgne shape: {df.shape}')

# 1. Duplikaadid
dupes_before = df.duplicated(subset='sale_id').sum()
print(f'\nDuplikaadid (sale_id alusel): {dupes_before}')
df = df.drop_duplicates(subset='sale_id')
print(f'Pärast duplikaatide eemaldamist: {df.shape}')

In [ ]:
# 2. NULL väärtused
print('NULL väärtused veergude kaupa:')
print(df.isnull().sum())

# Eemalda kriitiliste veergude NULLid
df = df.dropna(subset=['customer_id', 'sale_date', 'total_price'])
print(f'\nPärast NULL eemaldamist: {df.shape}')

In [ ]:
# 3. Kuupäevade parsimine (mixed formats: YYYY-MM-DD ja DD/MM/YYYY)
def parse_date_mixed(d):
    """Parsib kuupäevi kahes formaadis: YYYY-MM-DD ja DD/MM/YYYY"""
    try:
        if '/' in str(d):
            return pd.to_datetime(d, format='%d/%m/%Y')
        return pd.to_datetime(d, format='%Y-%m-%d')
    except:
        return pd.NaT

df['sale_date'] = df['sale_date'].apply(parse_date_mixed)
df = df.dropna(subset=['sale_date'])
print(f'sale_date dtype: {df["sale_date"].dtype}')
print(f'Kuupäevavahemik: {df["sale_date"].min().date()} kuni {df["sale_date"].max().date()}')

In [ ]:
# 4. Negatiivsed ja nullhinnad (tagastused) — eemaldame RFM jaoks
neg_count = (df['total_price'] <= 0).sum()
print(f'Negatiivsed/null total_price: {neg_count}')
df = df[df['total_price'] > 0]

# 5. Tuleviku kuupäevad — eemaldame
reference_date = pd.Timestamp('2025-02-28')
future = (df['sale_date'] > reference_date).sum()
print(f'Tuleviku kuupäevad: {future}')
df = df[df['sale_date'] <= reference_date]

# Puhastusraport
print('\n=== PUHASTUSRAPORT ===')
print(f'Lõplik shape:        {df.shape}')
print(f'Unikaalseid kliente: {df["customer_id"].nunique()}')
print(f'Kuupäevavahemik:     {df["sale_date"].min().date()} kuni {df["sale_date"].max().date()}')
print(f'Käive kokku:         {df["total_price"].sum():,.2f} EUR')
print(f'Keskmine tellimus:   {df["total_price"].mean():,.2f} EUR')

---
## Roll C: RFM Analysis — Recency, Frequency, Monetary

In [ ]:
# Viitekuupäev
today = pd.Timestamp('2025-02-28')

# Recency: päevad viimasest ostust
recency = df.groupby('customer_id')['sale_date'].max().reset_index()
recency.columns = ['customer_id', 'last_purchase']
recency['recency_days'] = (today - recency['last_purchase']).dt.days

# Frequency: ostude arv
frequency = df.groupby('customer_id')['sale_id'].count().reset_index()
frequency.columns = ['customer_id', 'frequency']

# Monetary: kogukulutus
monetary = df.groupby('customer_id')['total_price'].sum().reset_index()
monetary.columns = ['customer_id', 'monetary_value']

# Liida RFM üheks tabeliks
rfm = recency.merge(frequency, on='customer_id').merge(monetary, on='customer_id')
print(f'RFM tabel: {rfm.shape}')
print(rfm.describe().round(2))

In [ ]:
# RFM skoorid kvintiilide alusel (1-5)
# Recency: madal recency (hiljuti ostis) = kõrge skoor (5) → VASTUPIDINE
rfm['R_score'] = pd.qcut(rfm['recency_days'], 5, labels=[5, 4, 3, 2, 1]).astype(int)

# Frequency ja Monetary: suurem = kõrgem skoor
rfm['F_score'] = pd.qcut(
    rfm['frequency'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5]
).astype(int)
rfm['M_score'] = pd.qcut(
    rfm['monetary_value'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5]
).astype(int)

# Koondskoor
rfm['RFM_Score'] = rfm['R_score'] + rfm['F_score'] + rfm['M_score']

print('RFM skooride jaotus:')
print(rfm[['R_score', 'F_score', 'M_score', 'RFM_Score']].describe().round(2))

In [ ]:
# Segmenteerimine RFM skoori alusel
def assign_segment(score):
    if score >= 13: return 'VIP Champions'
    elif score >= 10: return 'Loyal'
    elif score >= 7: return 'Potential'
    elif score >= 4: return 'At Risk'
    else: return 'Lost'

rfm['Segment'] = rfm['RFM_Score'].apply(assign_segment)

# Segmentide kokkuvõte
segment_summary = rfm.groupby('Segment').agg(
    klientide_arv=('customer_id', 'count'),
    kogukäive=('monetary_value', 'sum'),
    kesk_recency=('recency_days', 'mean'),
    kesk_frequency=('frequency', 'mean'),
    kesk_monetary=('monetary_value', 'mean')
).round(2)

total = rfm.shape[0]
segment_summary['osakaal_%'] = (segment_summary['klientide_arv'] / total * 100).round(1)
segment_summary['käive_osakaal_%'] = (
    segment_summary['kogukäive'] / segment_summary['kogukäive'].sum() * 100
).round(1)

print('=== SEGMENTIDE KOKKUVÕTE ===')
print(segment_summary[['klientide_arv', 'osakaal_%', 'kogukäive', 'käive_osakaal_%', 'kesk_monetary']]
      .sort_values('kogukäive', ascending=False))

---
## Roll D: Visualization — 3 diagrammi + äritõlgendus

In [ ]:
# Diagramm 1: Segmentide klientide arv (tulpdiagramm)
seg_order = ['VIP Champions', 'Loyal', 'Potential', 'At Risk', 'Lost']
seg_colors = {
    'VIP Champions': '#009B8D',
    'Loyal': '#1A7A74',
    'Potential': '#4DB8B0',
    'At Risk': '#F4A261',
    'Lost': '#E76F51'
}

seg_plot = rfm.groupby('Segment').agg(
    klientide_arv=('customer_id', 'count'),
    kogukäive=('monetary_value', 'sum')
).reset_index()
seg_plot['Segment'] = pd.Categorical(seg_plot['Segment'], categories=seg_order, ordered=True)
seg_plot = seg_plot.sort_values('Segment')

fig1 = px.bar(
    seg_plot,
    x='Segment',
    y='klientide_arv',
    color='Segment',
    color_discrete_map=seg_colors,
    title='UrbanStyle Kliendisegmentide jaotus (RFM)',
    labels={'klientide_arv': 'Klientide arv', 'Segment': 'Segment'},
    text='klientide_arv'
)
fig1.update_traces(textposition='outside')
fig1.update_layout(
    showlegend=False,
    plot_bgcolor='white',
    yaxis=dict(gridcolor='#f0f0f0'),
    font=dict(family='Arial')
)
fig1.add_annotation(
    x='VIP Champions', y=388 + 30,
    text='39.4% käibest!<br>vaid 15.3% klientidest',
    showarrow=True, arrowhead=2,
    ax=60, ay=-40,
    font=dict(color='#009B8D', size=11)
)
fig1.add_annotation(
    x='At Risk', y=612 + 30,
    text='612 klienti ohus!<br>Win-back vajalik',
    showarrow=True, arrowhead=2,
    ax=60, ay=-40,
    font=dict(color='#E76F51', size=11)
)
fig1.show()

In [ ]:
# Diagramm 2: Hajuvusdiagramm — recency vs monetary, segmendi järgi
fig2 = px.scatter(
    rfm,
    x='recency_days',
    y='monetary_value',
    color='Segment',
    size='frequency',
    color_discrete_map=seg_colors,
    title='UrbanStyle Kliendisegmendid — Recency vs Monetary',
    labels={
        'recency_days': 'Päevi viimasest ostust (vähem = parem)',
        'monetary_value': 'Kogukulutus (EUR)',
        'frequency': 'Ostude arv'
    },
    hover_data=['recency_days', 'frequency', 'monetary_value', 'RFM_Score'],
    opacity=0.7
)
fig2.update_layout(
    plot_bgcolor='white',
    xaxis=dict(gridcolor='#f0f0f0'),
    yaxis=dict(gridcolor='#f0f0f0'),
    font=dict(family='Arial')
)
# Viitejoon: keskmine monetary
avg_monetary = rfm['monetary_value'].mean()
fig2.add_hline(
    y=avg_monetary,
    line_dash='dash',
    line_color='gray',
    annotation_text=f'Keskmine: {avg_monetary:.0f} EUR',
    annotation_position='right'
)
fig2.show()

In [ ]:
# Diagramm 3: TOP 10 VIP Champions klienti
top_vip = rfm[rfm['Segment'] == 'VIP Champions'].nlargest(10, 'monetary_value').copy()
top_vip['rank'] = range(1, 11)
top_vip['label'] = 'Klient #' + top_vip['customer_id'].astype(str)

fig3 = px.bar(
    top_vip,
    x='monetary_value',
    y='label',
    orientation='h',
    title='TOP 10 VIP Champions — Kogukulutus (EUR)',
    labels={
        'monetary_value': 'Kogukulutus (EUR)',
        'label': 'Klient'
    },
    color='monetary_value',
    color_continuous_scale=['#4DB8B0', '#009B8D', '#1A1A2E'],
    text='monetary_value'
)
fig3.update_traces(
    texttemplate='€%{text:,.0f}',
    textposition='outside'
)
fig3.update_layout(
    yaxis=dict(autorange='reversed'),
    plot_bgcolor='white',
    xaxis=dict(gridcolor='#f0f0f0'),
    showlegend=False,
    coloraxis_showscale=False,
    font=dict(family='Arial')
)
fig3.add_annotation(
    x=top_vip['monetary_value'].max(),
    y=0,
    text='Top klient: 27 921 €',
    showarrow=True, arrowhead=2,
    ax=-80, ay=-30,
    font=dict(color='#009B8D', size=11)
)
fig3.show()

In [ ]:
# Eksport CSV failina turundusmeeskonnale
rfm_export = rfm[['customer_id', 'recency_days', 'frequency', 'monetary_value',
                   'R_score', 'F_score', 'M_score', 'RFM_Score', 'Segment']]
rfm_export.to_csv('rfm_segments.csv', index=False)
print('rfm_segments.csv salvestatud!')
print(f'Ridu: {rfm_export.shape[0]} | Veerge: {rfm_export.shape[1]}')

---
## Kokkuvõte: Soovitused Markole

### RFM analüüsi peamised leiud

| Segment | Kliente | Osakaal | Käive | Käive % |
|---------|---------|---------|-------|--------|
| VIP Champions | 388 | 15.3% | 1 053 783 € | 39.4% |
| Loyal | 674 | 26.5% | 820 167 € | 30.6% |
| Potential | 705 | 27.8% | 513 489 € | 19.2% |
| At Risk | 612 | 24.1% | 259 032 € | 9.7% |
| Lost | 161 | 6.3% | 30 379 € | 1.1% |

### 3 konkreetset soovitust Markole

**1. VIP programm (388 klienti — 39.4% käibest)**  
Need 15.3% klientidest genereerivad peaaegu 40% käibest. Loo neile eksklusiivne VIP programm: early access uutele kollektsioonidele, personaliseeritud allahindlused, prioriteetne klienditeenindus. Iga VIP kliendi kaotamine maksab keskmiselt ~2 717 EUR aastas.

**2. Win-back kampaania (612 At Risk klienti — 9.7% käibest)**  
612 klienti pole hiljuti ostnud, kuid on varem kulutanud. Saada personaliseeritud e-mail sooduskoodiga (10-15%). Isegi 20% konversioon tooks tagasi ~122 klienti ja ~50 000 EUR lisatulu.

**3. Potential segmendi kasvatamine (705 klienti)**  
Potential kliendid on keskmised ostjad — nende üleviimine Loyal segmenti on odavam kui uute klientide hankimine. Soovitus: cross-sell kampaaniad ja lojaalsuspunktide süsteem.